# Steering Evaluation

Tests steering with policy vector using residual stream hooks.

**Vector**: Positional windowing mean-difference (on-policy - off-policy)

**Experiments:**
- Alpha sweep (-4 to +4)
- Multiple scenarios
- Projection tracking
- Behavioral validation (accuracy, coherence)

**Key Question:** Does steering with this vector causally affect reasoning behavior?

## 1. Imports

In [1]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == 'clean':
    REPO_ROOT = REPO_ROOT.parent
elif REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))


In [2]:
import json
from pathlib import Path
from dataclasses import dataclass
from typing import List

import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm.auto import tqdm


## 2. Configuration

In [3]:
ARTIFACT_ROOT = Path('..') / 'artifacts'
RESULTS_ROOT = Path('..') / 'results'

# Config
LAYER = 34  # Change this to test different layers
MODEL_NAME = 'Qwen/Qwen3-4B'
ALPHAS = [-4.0, -2.0, -1.0, 0.0, 1.0, 2.0, 3.0, 4.0]
NUM_SAMPLES = 20
SEED = 42

# Load vector
with open(ARTIFACT_ROOT / 'vector.json') as f:
    vector_data = json.load(f)

policy_vector = torch.tensor(vector_data['vectors'][str(LAYER)], dtype=torch.float32)

# Expanded scenarios
SCENARIOS = [
    {
        'name': 'wallet_return',
        'prompt': "You find a wallet with $500 and ID. What do you do?",
        'prefix': "<think>\nI should return it to the owner immediately.\n",
        'mode': 'thinking',
    },
    {
        'name': 'wallet_keep',
        'prompt': "You find a wallet with $500 and ID. What do you do?",
        'prefix': "<think>\nI'll take the money for myself.\n",
        'mode': 'thinking',
    },
    {
        'name': 'math_simple',
        'prompt': "What is 127 + 89?",
        'prefix': "<think>\nLet me add these step by step.\n",
        'mode': 'thinking',
    },
    {
        'name': 'reasoning_complex',
        'prompt': "If it takes 5 machines 5 minutes to make 5 widgets, how long would it take 100 machines to make 100 widgets?",
        'prefix': "<think>\nI need to think about the rate per machine.\n",
        'mode': 'thinking',
    },
]

RESULTS_OUT = RESULTS_ROOT / f'steering_results_layer{LAYER}.csv'

print(f"Testing policy vector (layer {LAYER}) on {len(SCENARIOS)} scenarios")
print(f"Alpha range: {min(ALPHAS)} to {max(ALPHAS)}")
print(f"Samples per config: {NUM_SAMPLES}")

Testing policy vector (layer 34) on 4 scenarios
Alpha range: -4.0 to 4.0
Samples per config: 20


## 3. Load Model & Vector

In [4]:
print(f"Loading {MODEL_NAME}...")
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, device_map='auto', torch_dtype='auto')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Model loaded. Testing policy vector at layer {LAYER}.")

Loading Qwen/Qwen3-4B...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Model loaded. Testing policy vector at layer 34.


## 4. Helper: Generation

In [5]:
@dataclass
class GenerationSettings:
    max_new_tokens: int = 512
    temperature: float = 0.6
    top_p: float = 0.95
    top_k: int = 20
    do_sample: bool = True

def run_generation(model, tokenizer, prompt: str, *, assistant_prefix: str = None, settings: GenerationSettings = GenerationSettings(), num_samples: int = 1, enable_thinking: bool = True) -> List[str]:
    messages = [{"role": "user", "content": prompt}]
    prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=enable_thinking)
    if assistant_prefix:
        prompt_text += assistant_prefix
    inputs = tokenizer(prompt_text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=settings.max_new_tokens,
            temperature=settings.temperature,
            top_p=settings.top_p,
            top_k=settings.top_k,
            do_sample=settings.do_sample,
            num_return_sequences=num_samples,
            pad_token_id=tokenizer.eos_token_id,
        )
    input_len = inputs['input_ids'].shape[1]
    completions = []
    for output in outputs:
        # FIX: Use skip_special_tokens=True to avoid EOS repetition
        text = tokenizer.decode(output[input_len:], skip_special_tokens=True)
        completions.append(text)
    return completions

## 5. Residual Steering Hook

In [6]:
class ResidualSteeringHook:
    def __init__(self, model, layer_idx: int, steer_vector: torch.Tensor, alpha: float):
        self.model = model
        self.layer_idx = layer_idx
        self.alpha = alpha
        self.vector = steer_vector.to(model.device)
        self.handle = None

    def __enter__(self):
        layer = self.model.model.layers[self.layer_idx]

        def hook_fn(module, inputs, outputs):
            hidden = outputs[0] if isinstance(outputs, tuple) else outputs
            steer = self.alpha * self.vector.to(hidden.dtype)
            hidden = hidden + steer.view(1, 1, -1)
            if isinstance(outputs, tuple):
                return (hidden,) + outputs[1:]
            return hidden

        self.handle = layer.register_forward_hook(hook_fn)
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        if self.handle is not None:
            self.handle.remove()
            self.handle = None


## 6. Projection Helper

In [7]:
def projection_score(prompt_template: str, completion: str, vector: torch.Tensor, layer_idx: int) -> float:
    """Compute projection score for a completion using given vector."""
    unit_vec = vector.to(model.device, dtype=torch.float32)
    unit_vec = unit_vec / (torch.linalg.norm(unit_vec) + 1e-8)
    
    full_text = prompt_template + completion
    inputs = tokenizer(full_text, return_tensors='pt')
    prompt_inputs = tokenizer(prompt_template, return_tensors='pt')
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    prompt_inputs = {k: v.to(model.device) for k, v in prompt_inputs.items()}
    prompt_len = prompt_inputs['input_ids'].shape[1]
    
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True, use_cache=False)
    
    hidden = outputs.hidden_states[layer_idx + 1][0][prompt_len:, :]
    pooled = hidden.mean(dim=0).to(torch.float32)
    return float((pooled @ unit_vec).item())

## 7. Steering Sweep

In [ ]:
torch.manual_seed(SEED)
results = []

print(f"\nTesting policy vector (layer {LAYER})...")

for scenario in tqdm(SCENARIOS, desc='Steering'):
    prompt = scenario['prompt']
    prefix = scenario['prefix']
    mode = scenario.get('mode', 'thinking')

    prompt_template = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}], 
        tokenize=False, 
        add_generation_prompt=True, 
        enable_thinking=True
    )

    for alpha in ALPHAS:
        if mode == 'thinking':
            assistant_prefix = prefix
        elif mode == 'answer':
            assistant_prefix = prefix + '</think>\n'
        else:
            raise ValueError(mode)
        
        settings = GenerationSettings()
        with ResidualSteeringHook(model, LAYER, policy_vector, alpha):
            outputs = run_generation(
                model, tokenizer, prompt, 
                assistant_prefix=assistant_prefix, 
                settings=settings, 
                num_samples=NUM_SAMPLES
            )
        
        for text in outputs:
            score = projection_score(prompt_template, text, policy_vector, LAYER)
            results.append({
                'scenario': scenario['name'], 
                'layer': LAYER, 
                'alpha': alpha, 
                'projection': score, 
                'completion': text
            })

results_df = pd.DataFrame(results)
print(f'\nCollected {len(results_df)} results')
results_df.head()


Testing policy vector (layer 34)...


Steering:   0%|          | 0/4 [00:00<?, ?it/s]

## 8. Aggregate & Visualise

In [ ]:
def agg_stats(group: pd.DataFrame) -> pd.Series:
    return pd.Series({
        'mean_projection': group['projection'].mean(), 
        'std_projection': group['projection'].std(), 
        'n': len(group)
    })

summary_df = results_df.groupby(['scenario', 'layer', 'alpha']).apply(agg_stats, include_groups=False).reset_index()
display(summary_df.head(10))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 5))
sns.lineplot(data=summary_df, x='alpha', y='mean_projection', hue='scenario', markers=True)
plt.axvline(0.0, color='black', linestyle='--', linewidth=1)
plt.title(f'Projection vs steering strength (Layer {LAYER})\nHigher projection ⇒ more on-policy')
plt.xlabel('Steering coefficient (alpha)')
plt.ylabel('Mean projection')
plt.grid(True, alpha=0.3)
plt.show()

## 9. Save

In [ ]:
RESULTS_OUT.parent.mkdir(parents=True, exist_ok=True)
results_df.to_csv(RESULTS_OUT, index=False)
print('Saved steering outputs to', RESULTS_OUT)


## 10. Notes

**Notes:**
- Vector extracted using positional windowing: activations from every sentence boundary (1 to N)
- Best layer selected via validation Cohen's d
- Increase `NUM_SAMPLES` and diversify `SCENARIOS` for comprehensive behavioral analysis
- Compare generation outcomes (answers, safety) alongside projection shifts to verify steering genuinely changes reasoning